# ⚡ CircuitAI — Colab GPU Backend
**Step 1**: Runtime → Change runtime type → T4 or L4 GPU

**Step 2**: Run cells in order.

**Step 3**: Wait for '✅ All models ready' log, then copy the `trycloudflare.com` URL into your local frontend.

In [ ]:
# ── Cell 0: Mount Drive & Set Working Directory ─────────────────
import sys, os
from google.colab import drive

drive.mount('/content/drive')

# ⚠️ Update this path if your folder is in a different location
BACKEND_DIR = '/content/drive/MyDrive/knowledge_graph/backend'

os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir()}')

In [ ]:
# ── Cell 1: Install Cloudflared (only system dep needed) ───────
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb
print('Cloudflared installed ✓')

In [ ]:
# ── Cell 2: Install Python dependencies ────────────────────────
!pip install -r requirements.txt -q
print('Python deps installed ✓')

In [ ]:
# ── Cell 3: Start FastAPI server + Cloudflare Tunnel ───────────
# The FastAPI lifespan event will automatically pre-load:
#   • Qwen2.5-VL-7B-Instruct (4-bit) into GPU
#   • BGE-M3 embedding model
# Wait for '[startup] ✅ All models ready' before uploading PDFs.

import subprocess, threading, time, uvicorn
from main import app

def start_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for model loading (usually 60-90s on T4)
print('Server starting... waiting for models to load.')
print('Watch for: [startup] ✅ All models ready')
time.sleep(5)  # give uvicorn time to bind before starting tunnel

# Start Cloudflare tunnel
print('Starting Cloudflare tunnel...')
p = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        tokens = line.split()
        public_url = next((t for t in tokens if 'trycloudflare.com' in t), None)
        if public_url:
            print('\n' + '='*60)
            print(f'  🔥 PUBLIC URL: {public_url}')
            print('  Paste into CircuitAI frontend > Connection field')
            print('  ⏳ Then wait for models to finish loading before uploading PDF.')
            print('='*60)
        break